# [PlotPot](https://github.com/cryotud/plotpot) SEC-MALS plotting notebook 

Creates publication-quality SEC-MALS chromatograms from ASTRA exports.

**Input format:** Tab-separated ASTRA export (`.txt`). Number format is auto-detected — both European (comma as decimal) and standard (period as decimal) are supported.

**Expected column layout (repeating per run):**
```
Columns 0–7 per run:  (vol, UV), (vol, dRI), (vol, UV2), (vol, Mw_Da)
```
Runs are listed in file order. BSA is auto-detected by name. Mw column contains data only inside the integration window set in ASTRA.

**Workflow** [run cells top to bottom]:
1. **Dependencies** & **Imports** — run once.
2. **Upload**: file picker appears; select your ASTRA `.txt` export.
3. **Labels & options**: select runs to plot, set volume windows; re-run to update.
4. **Overview**: all runs, UV and dRI+Mw sanity check.
5. **Publication plot**: selected runs + BSA, will plot publication-quality panels.
6. **Save & download**: writes PDF + PNG and downloads to your machine.

In [ ]:
#@title Step 1 · Install & verify dependencies { display-mode: "form" }
import importlib, subprocess, sys

_required = ['numpy', 'pandas', 'matplotlib']
_missing  = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _missing)
    print(f'Installed: {_missing}')
else:
    print('All dependencies present:', _required)

In [ ]:
#@title Step 2 · Imports & plot style { display-mode: "form" }
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

In [ ]:
#@title Step 3 · Upload data { display-mode: "form" }
from google.colab import files as _colab_files

print('A file picker will appear below — select your ASTRA export:')
_uploaded = _colab_files.upload()

_fname    = next(iter(_uploaded))
DATA_FILE = Path(_fname)


def _detect_number_format(sample_parts):
    """
    Detect whether numbers use European (comma=decimal, period=thousands)
    or standard (period=decimal, comma=thousands) format.
    """
    for p in sample_parts:
        p = p.strip()
        if not p or p in ('nan', 'NaN', 'None'):
            continue
        if p.startswith(',') or p.startswith('-,'):
            return 'european'
        if ',' in p and '.' in p:
            return 'european' if p.rindex(',') > p.rindex('.') else 'standard'
    return 'standard'


def _make_number_parser(fmt):
    if fmt == 'european':
        def _parse(s):
            s = str(s).strip()
            if s in ('', 'nan', 'NaN', 'None'):
                return np.nan
            s = s.replace('.', '').replace(',', '.')
            try:
                return float(s)
            except ValueError:
                return np.nan
    else:
        def _parse(s):
            s = str(s).strip()
            if s in ('', 'nan', 'NaN', 'None'):
                return np.nan
            s = s.replace(',', '')
            try:
                return float(s)
            except ValueError:
                return np.nan
    return _parse


def load_astra_multi(src):
    """
    Parse an ASTRA multi-run export (.txt).

    Header alternates: 'volume (mL)' | 'RunLabel[Session]' | ...
    Each run occupies 4 consecutive column pairs: UV, dRI, UV2, Mw_Da.
    Number format (European or standard) is auto-detected from the first data row.

    Returns (runs_order, runs_data, number_fmt):
      runs_order  -- list of run labels in file order
      runs_data   -- dict {label: DataFrame} with columns
                     uv_vol, uv, ri_vol, ri, uv2_vol, uv2, mw_vol, mw_da
      number_fmt  -- 'european' or 'standard'
    """
    PAIRS = ['uv', 'ri', 'uv2', 'mw']

    if hasattr(src, 'read'):
        fh, _close = src, False
    else:
        fh, _close = open(src, encoding='utf-8'), True

    runs_order, col_map, pair_count = [], {}, {}
    raw_rows = []
    number_fmt = 'standard'
    parse = None

    try:
        for lineno, line in enumerate(fh):
            parts = line.rstrip('\n').split('\t')
            if lineno == 0:
                for i in range(0, len(parts) - 1, 2):
                    label = parts[i + 1].split('[')[0].strip()
                    if label not in col_map:
                        col_map[label] = {}
                        runs_order.append(label)
                        pair_count[label] = 0
                    idx = pair_count[label]
                    if idx < len(PAIRS):
                        col_map[label][PAIRS[idx]] = i
                    pair_count[label] += 1
                continue
            if parse is None:
                number_fmt = _detect_number_format(parts)
                parse = _make_number_parser(number_fmt)
            raw_rows.append([parse(p) for p in parts])
    finally:
        if _close:
            fh.close()

    raw = pd.DataFrame(raw_rows)
    runs_data = {}
    for label in runs_order:
        d = {}
        for pair, vcol in col_map[label].items():
            scol = vcol + 1
            sig  = 'mw_da' if pair == 'mw' else pair
            d[f'{pair}_vol'] = raw.iloc[:, vcol].values if vcol < raw.shape[1] else np.full(len(raw), np.nan)
            d[sig]           = raw.iloc[:, scol].values if scol < raw.shape[1] else np.full(len(raw), np.nan)
        runs_data[label] = pd.DataFrame(d)

    return runs_order, runs_data, number_fmt


runs_order, runs_data, _fmt = load_astra_multi(
    io.StringIO(_uploaded[_fname].decode('utf-8'))
)

bsa_run     = next((r for r in runs_order if 'bsa' in r.lower()), runs_order[0])
sample_runs = [r for r in runs_order if r != bsa_run]

print(f'Loaded {_fname!r}  ({len(runs_data[runs_order[0]]):,} rows)')
print(f'Number format detected: {_fmt}\n')
print(f'{len(runs_order)} runs found  (BSA auto-detected: {bsa_run!r})\n')
for label in runs_order:
    tag = ' <- BSA' if label == bsa_run else ''
    mw  = runs_data[label]['mw_da'].dropna()
    mws = f'  Mw: {mw.min()/1e3:.1f}-{mw.max()/1e3:.1f} kDa (n={len(mw)})' if len(mw) else '  Mw: none'
    print(f'  * {label!r}{tag}{mws}')
print()
print('Paste these labels into SELECTED_RUNS in Step 4:')
print('  ' + ', '.join(sample_runs))

In [ ]:
#@title Step 4 · Labels & options { display-mode: "form" }
#@markdown Edit fields below, then **Run** (Shift+Enter).
#@markdown Copy run labels from the Step 3 output.

#@markdown **Run selection** — comma-separated run labels; leave empty to plot all sample runs.
SELECTED_RUNS = "" #@param {type:"string"}
#@markdown **Run display labels** — comma-separated names shown in plots (same order as SELECTED_RUNS); leave empty to use file labels.
RUN_LABELS_STR = "" #@param {type:"string"}
#@markdown **BSA run** — leave empty to use the auto-detected BSA run.
BSA_RUN_OVERRIDE = "" #@param {type:"string"}
BSA_LABEL = "BSA" #@param {type:"string"}
SHOW_BSA  = True  #@param {type:"boolean"}

#@markdown ---
#@markdown **Sample volume window (mL)** — one value applied to all runs, or comma-separated per run.
SAMPLE_VOL_MIN_STR = "8.0"  #@param {type:"string"}
SAMPLE_VOL_MAX_STR = "20.0" #@param {type:"string"}
#@markdown **BSA volume window (mL)**
BSA_VOL_MIN = 13.0  #@param {type:"number"}
BSA_VOL_MAX = 17.0  #@param {type:"number"}

#@markdown ---
#@markdown **Molar mass axis** — log scale, shared across all panels.
MW_YLIM_MIN  = 1     #@param {type:"number"}
MW_YLIM_MAX  = 10000 #@param {type:"number"}
#@markdown **Mw filter** — hide dots where dRI < this fraction of peak height.
MW_RI_THRESHOLD = 0.10 #@param {type:"slider", min:0.0, max:0.5, step:0.01}

# ── Resolve selection ──────────────────────────────────────────────────────────
_active_bsa = BSA_RUN_OVERRIDE.strip() if BSA_RUN_OVERRIDE.strip() else bsa_run

if SELECTED_RUNS.strip():
    _sel     = [s.strip() for s in SELECTED_RUNS.split(',') if s.strip() in runs_data]
    _missing = [s.strip() for s in SELECTED_RUNS.split(',') if s.strip() not in runs_data]
    if _missing:
        print(f'WARNING: runs not found (check spelling): {_missing}')
else:
    _sel = list(sample_runs)

def _parse_vol_str(s, n):
    vals = [float(x.strip()) for x in str(s).split(',') if x.strip()]
    if len(vals) == 1:
        return vals * n
    return (vals + [vals[-1]] * n)[:n]

n_sel        = len(_sel)
VOL_MINS     = _parse_vol_str(SAMPLE_VOL_MIN_STR, n_sel)
VOL_MAXS     = _parse_vol_str(SAMPLE_VOL_MAX_STR, n_sel)
MW_YLIM      = (MW_YLIM_MIN, MW_YLIM_MAX)
BSA_VOL_RANGE = (BSA_VOL_MIN, BSA_VOL_MAX)

_custom = [l.strip() for l in RUN_LABELS_STR.split(',')]
_labels = [_custom[i] if i < len(_custom) and _custom[i] else r
           for i, r in enumerate(_sel)]

print(f'BSA run  : {_active_bsa!r}  (show = {SHOW_BSA})')
print(f'Selected runs ({n_sel}):')
for r, lbl, vmin, vmax in zip(_sel, _labels, VOL_MINS, VOL_MAXS):
    mw  = runs_data[r]['mw_da'].dropna() if r in runs_data else pd.Series(dtype=float)
    mws = f'median {mw.median()/1e3:.0f} kDa' if len(mw) else 'no Mw'
    tag = f' (label: {lbl!r})' if lbl != r else ''
    print(f'  • {r!r}{tag}  vol {vmin}–{vmax} mL  ({mws})')
if SHOW_BSA:
    print(f'BSA panel: vol {BSA_VOL_MIN}–{BSA_VOL_MAX} mL')
print(f'Mw axis  : {MW_YLIM_MIN}–{MW_YLIM_MAX} kDa (log)')

In [ ]:
#@title Step 5 · Overview (all runs) { display-mode: "form" }
n_runs = len(runs_order)
fig_ov, axes_ov = plt.subplots(n_runs, 2, figsize=(12, 3.5 * n_runs), squeeze=False, layout='constrained')

for row, label in enumerate(runs_order):
    df_r   = runs_data[label]
    ax_uv  = axes_ov[row, 0]
    ax_dri = axes_ov[row, 1]

    # UV trace
    uv = df_r[['uv_vol', 'uv']].dropna()
    ax_uv.plot(uv['uv_vol'], uv['uv'], color='steelblue', lw=0.8)
    ax_uv.set_title(f'{label} — UV', fontsize=10)
    ax_uv.set_xlabel('Volume (mL)')
    ax_uv.set_ylabel('UV (AU)')
    ax_uv.spines['top'].set_visible(False)
    ax_uv.spines['right'].set_visible(False)

    # dRI + Mw overlay
    ri = df_r[['ri_vol', 'ri']].dropna()
    ax_dri.plot(ri['ri_vol'], ri['ri'], color='forestgreen', lw=0.8)
    ax_dri.set_title(f'{label} — dRI + Mw', fontsize=10)
    ax_dri.set_xlabel('Volume (mL)')
    ax_dri.set_ylabel('dRI')
    ax_dri.spines['top'].set_visible(False)

    ax_mw2 = ax_dri.twinx()
    mw = df_r[['mw_vol', 'mw_da']].dropna()
    if len(mw):
        ax_mw2.scatter(mw['mw_vol'], mw['mw_da'] / 1e3, color='firebrick', s=3, zorder=5, alpha=0.8)
    ax_mw2.set_yscale('log')
    ax_mw2.set_ylim(*MW_YLIM)
    ax_mw2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
    ax_mw2.set_ylabel('Molar mass (kDa)', color='firebrick')
    ax_mw2.tick_params(axis='y', labelcolor='firebrick')

fig_ov.suptitle(f'{DATA_FILE.stem} — overview ({n_runs} runs)', fontsize=13)
plt.show()

In [ ]:
#@title Step 6 · Publication plot { display-mode: "form" }
#@markdown *Adjust selection and windows in Step 4, then run this cell.*

from matplotlib.transforms import blended_transform_factory as _btf

SAMPLE_COLORS = ['#1a4f8a', '#c0392b', '#27ae60', '#8e44ad', '#d35400']
C_BSA = '#888888'
C_MW  = '#f0c106'

show_bsa_panel = SHOW_BSA and bool(_active_bsa) and _active_bsa in runs_data
n_panels       = len(_sel) + (1 if show_bsa_panel else 0)

if n_panels == 0:
    print('No runs to plot — check SELECTED_RUNS in Step 4.')
else:
    fig, axes_p = plt.subplots(1, n_panels, figsize=(4.5 * n_panels, 3.8),
                                layout='constrained')
    if n_panels == 1:
        axes_p = [axes_p]

    def _pub_panel(ax_mw, df_r, vmin, vmax, label, color):
        ax_dri = ax_mw.twinx()

        # Normalized dRI (right axis)
        ri = df_r[['ri_vol', 'ri']].dropna()
        m  = (ri['ri_vol'] >= vmin) & (ri['ri_vol'] <= vmax)
        v, s = ri.loc[m, 'ri_vol'].values, ri.loc[m, 'ri'].values
        s_norm = s / s.max() if len(s) and s.max() > 0 else s
        ax_dri.plot(v, s_norm, color=color, lw=1.5, label=label)
        ax_dri.set_ylim(-0.06, 1.05)
        ax_dri.set_ylabel('Normalized dRI', fontsize=11)

        # Mw scatter (left axis)
        mw = df_r[['mw_vol', 'mw_da']].dropna()
        mw = mw[(mw['mw_vol'] >= vmin) & (mw['mw_vol'] <= vmax)].copy()
        if len(mw) > 0 and len(v) > 0:
            ri_at = np.interp(mw['mw_vol'].values, v, s_norm)
            mw    = mw[ri_at >= MW_RI_THRESHOLD]

        mw_kda = mw['mw_da'].values / 1e3
        vol_mw = mw['mw_vol'].values
        ax_mw.scatter(vol_mw, mw_kda, color=C_MW, s=4, zorder=5, alpha=0.8,
                      label='Molar mass')
        if len(mw_kda):
            order = np.argsort(vol_mw)
            vw, mk = vol_mw[order], mw_kda[order]
            dv = np.diff(vw)
            med_step = float(np.median(dv[dv > 0])) if np.any(dv > 0) else 1.0
            splits = np.where(dv > 10 * med_step)[0] + 1
            for vg, mg in zip(np.split(vw, splits), np.split(mk, splits)):
                if not len(mg):
                    continue
                med = float(np.median(mg))
                ax_mw.axhline(med, color=C_MW, lw=0.8, ls='--', alpha=0.5)
                x_rel = (float(np.mean(vg)) - vmin) / (vmax - vmin) if vmax > vmin else 0.5
                x_ax, ha = (0.97, 'right') if x_rel < 0.5 else (0.03, 'left')
                ax_mw.text(
                    x_ax, med, f'{med:.0f} kDa',
                    transform=_btf(ax_mw.transAxes, ax_mw.transData),
                    color=C_MW, fontsize=9, va='center', ha=ha,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85),
                )

        ax_mw.set_xlim(vmin, vmax)
        ax_mw.set_ylim(*MW_YLIM)
        ax_mw.set_yscale('log')
        ax_mw.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
        ax_mw.set_xlabel('Elution volume (mL)', fontsize=11)
        ax_mw.set_ylabel('Molar mass (kDa)', fontsize=11)
        ax_mw.set_title(label, fontsize=11, fontweight='bold')
        for spine in ax_mw.spines.values():
            spine.set_visible(True)
        for spine in ax_dri.spines.values():
            spine.set_visible(True)

        h1, l1 = ax_dri.get_legend_handles_labels()
        h2, l2 = ax_mw.get_legend_handles_labels()
        ax_mw.legend(h1 + h2, l1 + l2, loc='upper right', frameon=True,
                     facecolor='white', edgecolor='none', framealpha=0.85, fontsize=9)

    for i, (run_label, disp_label, vmin, vmax) in enumerate(zip(_sel, _labels, VOL_MINS, VOL_MAXS)):
        _pub_panel(axes_p[i], runs_data[run_label], vmin, vmax,
                   disp_label, SAMPLE_COLORS[i % len(SAMPLE_COLORS)])

    if show_bsa_panel:
        _pub_panel(axes_p[-1], runs_data[_active_bsa], BSA_VOL_MIN, BSA_VOL_MAX,
                   BSA_LABEL, C_BSA)

    plt.show()

In [ ]:
#@title Step 7 · Save & download { display-mode: "form" }
from google.colab import files as _colab_files

OUTPUT_STEM = DATA_FILE.stem

_outputs = []
for ext in ('pdf', 'png'):
    out = f'/content/{OUTPUT_STEM}_sec_mals.{ext}'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'Saved → {out}')
    _outputs.append(out)

print('\nStarting downloads...')
for out in _outputs:
    _colab_files.download(out)